# 02 — Anotação e divisão treino/validação/teste

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/02_anotacao_split/02_anotacao_split.ipynb)

**Objetivo:** anotar a classe **pluma**, dividir os dados **antes** de treinar e montar o dataset no formato YOLO.

**Escopo da professora:** seção 2 (avaliar em dados independentes) e seção 4 (divisão por sequência/local/data).

**Lê:** `01_dados/frames.csv` e `01_dados/frames/`; rótulos YOLO em `labels/`.

**Grava:** `split.csv`, `yolo_dataset/`, `dataset.yaml` e `dataset_B_publico.yaml`.

**GPU:** não


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "02_anotacao_split"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## 1. O que conta como pluma?
Escreva o critério **antes** de anotar e siga o mesmo nos dois anotadores. Edite esta célula:

- **Conta como pluma:** …
- **Não conta:** nuvem, neblina, vapor de água, poeira, …
- **Bordas difusas:** anotar até onde a pluma é visível com contraste razoável, caixa justa.
- **Pluma parcialmente oculta** (árvores, estruturas): …
- **Pluma que se mistura a nuvem:** …
- **Quadros sem pluma:** salvar o arquivo `.txt` **vazio** (são exemplos negativos úteis).

## 2. Anotação
Ferramentas: **Roboflow**, **CVAT** ou **Label Studio**. Exporte no formato **YOLO** (um `.txt` por imagem, linhas `classe cx cy w h`, normalizadas).

1. Anote uma amostra dos quadros (algumas centenas bem anotadas valem mais do que milhares mal anotados). Inclua quadros sem pluma.
2. Coloque os `.txt` em `<Drive>/02_anotacao_split/labels/`, com o **mesmo nome do quadro** (`seq01_000120.jpg` → `seq01_000120.txt`).
3. Para segmentação (etapa 04), anote também algumas **máscaras de referência** manuais em `<Drive>/04_sam_atributos/mascaras_gt/` (30 a 50 quadros bastam).

Opcional: pré-anotar com um modelo para só corrigir à mão (mais rápido).

In [ ]:
PRE_WEIGHTS = None     # ex.: caminho de pesos de fumaça já treinados. None = pular
frames_dir = stage_dir(cfg, "01_dados") / "frames"

if PRE_WEIGHTS:
    from ultralytics import YOLO
    m = YOLO(PRE_WEIGHTS)
    out = D / "labels_pre"; out.mkdir(exist_ok=True)
    for r in m.predict(source=str(frames_dir), conf=0.15, stream=True, verbose=False):
        lines = [f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for x, y, w, h in r.boxes.xywhn.tolist()]
        (out / (Path(r.path).stem + ".txt")).write_text("\n".join(lines))
    print("Pré-anotações em", out, "(revise e mova para labels/)")

## 3. Divisão sem vazamento
Regra do escopo: quadros muito parecidos da mesma sequência **não** podem ficar em treino e teste.

- **3 ou mais sequências/locais/datas:** divisão por grupo inteiro.
- **Uma única sequência:** blocos contíguos no tempo (treino → validação → teste) com um intervalo descartado entre eles.
- **Duas sequências:** defina à mão em `MANUAL` (por exemplo, uma sequência inteira como teste).

O arquivo `split.csv` é **congelado**: depois de criado, não é sobrescrito sem `FORCE = True`.

In [ ]:
import pandas as pd
from plumas import split as sp

FORCE = False
MANUAL = {}      # ex.: {"seq01": "train", "seq02": "test"}

split_csv = D / "split.csv"
frames = pd.read_csv(stage_dir(cfg, "01_dados") / "frames.csv")

if split_csv.exists() and not FORCE:
    split = pd.read_csv(split_csv)
    print("split.csv já existe (congelado). Carregado.")
else:
    if MANUAL:
        split = frames.copy(); split["split"] = split["sequence"].map(MANUAL)
        assert split["split"].notna().all(), "MANUAL deve cobrir todas as sequências"
    elif frames["sequence"].nunique() >= 3:
        split = sp.group_split(frames, "sequence", cfg["split"]["ratios"], cfg["seed"])
    else:
        split = sp.block_split(frames, "t_s", cfg["split"]["ratios"], cfg["split"]["block_gap_frames"])
        print("Aviso: sequência única. Blocos contíguos com intervalo; o teste vem do final do vídeo.")
    if frames["sequence"].nunique() >= 3 or MANUAL:
        sp.check_no_leakage(split, "sequence")
    split.to_csv(split_csv, index=False)
    log_experiment(cfg, STAGE, {"decisao": "split criado", "n_por_split": split["split"].value_counts().to_dict(),
                                "seed": cfg["seed"], "ratios": cfg["split"]["ratios"]})

print(split.groupby(["split"]).agg(quadros=("frame", "size"), sequencias=("sequence", "nunique")))

## 4. Montar o dataset YOLO
Só entram quadros **anotados** (com `.txt` em `labels/`). A pasta `yolo_dataset/` é gerada e recriada a cada execução.

**Dados públicos (opcional):** se usar um dataset público de fumaça, adicione-o **só ao treino**, nunca ao teste.
Muitos têm mais de uma classe (fogo e fumaça): informe o id da classe de fumaça em `SMOKE_CLASS_ID` e as linhas de fogo são descartadas.
Além do `dataset.yaml` (variante C: públicos + próprios) é criado o `dataset_B_publico.yaml` (variante B: só públicos no treino).

In [ ]:
import shutil, yaml

labels_dir = D / "labels"
ds_dir = D / "yolo_dataset"
if ds_dir.exists():
    shutil.rmtree(ds_dir)                      # pasta gerada, recriada a cada execução
for sub in ("images", "labels"):
    for s in ("train", "val", "test"):
        (ds_dir / sub / s).mkdir(parents=True)

stats = []
for s in ("train", "val", "test"):
    n_img = n_pos = n_sem_rotulo = 0
    for fr in split.loc[split["split"] == s, "frame"]:
        lab = labels_dir / (Path(fr).stem + ".txt")
        if not lab.exists():
            n_sem_rotulo += 1
            continue
        shutil.copy2(frames_dir / fr, ds_dir / "images" / s / fr)
        shutil.copy2(lab, ds_dir / "labels" / s / lab.name)
        n_img += 1
        n_pos += int(lab.read_text().strip() != "")
    stats.append({"split": s, "anotados": n_img, "com_pluma": n_pos, "sem_pluma": n_img - n_pos, "sem_rotulo": n_sem_rotulo})
print(pd.DataFrame(stats))

In [ ]:
EXTRA_TRAIN_DIRS = []    # ex.: [Path("/content/drive/MyDrive/datasets/D-Fire/train")]  (com images/ e labels/)
SMOKE_CLASS_ID = 1       # id da classe "fumaça" nesses datasets (confira no data.yaml deles!)

ext_list = []
for ed in map(Path, EXTRA_TRAIN_DIRS):
    for img in (ed / "images").glob("*.*"):
        lab = ed / "labels" / (img.stem + ".txt")
        if not lab.exists():
            continue
        keep = []
        for line in lab.read_text().splitlines():
            p = line.split()
            if p and int(p[0]) == SMOKE_CLASS_ID:
                keep.append("0 " + " ".join(p[1:]))     # remapeia para a classe única "pluma"
        dst_img = ds_dir / "images" / "train" / ("ext_" + img.name)
        shutil.copy2(img, dst_img)
        (ds_dir / "labels" / "train" / ("ext_" + img.stem + ".txt")).write_text("\n".join(keep))
        ext_list.append(str(dst_img))
print("Imagens públicas adicionadas ao treino:", len(ext_list))

base = {"path": str(ds_dir), "val": "images/val", "test": "images/test",
        "names": {0: cfg["class_names"][0]}}
(D / "dataset.yaml").write_text(yaml.safe_dump({**base, "train": "images/train"}, allow_unicode=True))

if ext_list:                                    # variante B: só dados públicos no treino
    (ds_dir / "train_publico.txt").write_text("\n".join(ext_list))
    (D / "dataset_B_publico.yaml").write_text(
        yaml.safe_dump({**base, "train": str(ds_dir / "train_publico.txt")}, allow_unicode=True))
print("dataset.yaml gravado em", D)

In [ ]:
log_experiment(cfg, STAGE, {"decisao": "dataset YOLO montado", "estatisticas": stats,
                            "n_publicas_treino": len(ext_list), "extra_dirs": [str(p) for p in EXTRA_TRAIN_DIRS]})